In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image

from skimage.measure import regionprops, label, perimeter

In [2]:
# ============================================================
# Paths
# ============================================================

TRAIN_DIR = Path("train")

IMG_DIR = TRAIN_DIR
MASK_DIR = TRAIN_DIR / "masks"

CLASSIF_PATH = TRAIN_DIR / "classif.xlsx"

In [3]:
# ============================================================
# Loading Functions
# ============================================================

def load_image(path):
    return Image.open(path).convert("RGB")


def load_mask(path, image_size):
    mask = Image.open(path).convert("L")

    # On force le masque à avoir la même taille que l'image
    mask = mask.resize(image_size)

    mask = np.array(mask)

    return mask > 0

In [4]:
# ============================================================
# RGB Features
# ============================================================

def rgb_features(image, mask):

    image = np.array(image)

    pixels = image[mask]

    features = {}

    for i, color in enumerate(["R", "G", "B"]):

        values = pixels[:, i]

        features[f"{color}_min"] = np.min(values)
        features[f"{color}_max"] = np.max(values)
        features[f"{color}_mean"] = np.mean(values)
        features[f"{color}_median"] = np.median(values)
        features[f"{color}_std"] = np.std(values)

    return features

In [5]:
# ============================================================
# Mandatory Features
# ============================================================

def bug_ratio(mask):
    return np.sum(mask) / mask.size


def shape_symmetry(mask):

    flipped_mask = np.fliplr(mask)

    return np.mean(mask == flipped_mask)


def color_symmetry(image, mask):

    h, w, _ = image.shape

    mid = w // 2

    left_img = image[:, :mid]
    right_img = image[:, -mid:]

    right_img = np.fliplr(right_img)

    left_mask = mask[:, :mid]
    right_mask = mask[:, -mid:]

    right_mask = np.fliplr(right_mask)

    common_mask = left_mask & right_mask

    if np.sum(common_mask) == 0:
        return 0

    diff = np.abs(
        left_img[common_mask].astype(float)
        - right_img[common_mask].astype(float)
    )

    return np.mean(diff)

In [6]:
# ============================================================
# Additional Features
# ============================================================

def extra_shape_features(mask):

    features = {}

    labeled_mask = label(mask)

    props = regionprops(labeled_mask)

    if len(props) == 0:

        features["aspect_ratio"] = 0
        features["circularity"] = 0

        return features

    region = max(props, key=lambda r: r.area)

    min_row, min_col, max_row, max_col = region.bbox

    width = max_col - min_col
    height = max_row - min_row

    area = region.area

    perim = perimeter(mask)

    features["aspect_ratio"] = (
        width / height if height != 0 else 0
    )

    features["circularity"] = (
        4 * np.pi * area / (perim ** 2)
        if perim != 0
        else 0
    )

    return features

In [7]:
# ============================================================
# Feature Extraction
# ============================================================

all_features = []
skipped_images = []

for i in range(1, 251):

    img_path = IMG_DIR / f"{i}.JPG"
    mask_path = MASK_DIR / f"binary_{i}.tif"

    if not img_path.exists() or not mask_path.exists():
        skipped_images.append(i)
        continue

    image = load_image(img_path)
    mask = load_mask(mask_path, image.size)
    image = np.array(image)

    row = {
        "ID": i,
        "bug_ratio": bug_ratio(mask),
        "shape_symmetry": shape_symmetry(mask),
        "color_symmetry": color_symmetry(image, mask)
    }

    row.update(rgb_features(image, mask))
    row.update(extra_shape_features(mask))

    all_features.append(row)

features_df = pd.DataFrame(all_features)

print("Images processed:", len(features_df))
print("Skipped images:", skipped_images)

features_df.head()

Images processed: 249
Skipped images: [154]


,ID,bug_ratio,shape_symmetry,color_symmetry,R_min,R_max,R_mean,R_median,R_std,G_min,...,G_mean,G_median,G_std,B_min,B_max,B_mean,B_median,B_std,aspect_ratio,circularity
0,1,0.007428,0.994837,32.729621,5,208,68.085745,56.0,47.955399,3,...,54.882747,37.0,45.311197,0,193,39.891969,24.0,36.250747,1.223881,0.037764
1,2,0.008553,0.992870,34.742178,2,248,63.786498,55.0,42.153508,2,...,52.079650,34.0,42.866124,0,244,35.735147,19.0,34.617117,1.514286,0.039630
2,3,0.022093,0.964987,45.799129,3,255,107.176333,114.0,58.771359,0,...,87.794710,86.0,60.057604,0,255,63.383043,52.0,54.823596,1.060606,0.057050
3,4,0.013187,0.973769,46.718097,5,219,87.588243,88.0,46.322587,3,...,71.031852,61.0,46.085760,0,201,50.432535,36.0,37.445621,1.308511,0.035105
4,5,0.009165,0.988398,92.340692,6,255,123.177003,134.0,62.355944,0,...,100.897964,91.0,63.029355,0,245,80.946170,63.0,60.624962,1.200000,0.052363


In [8]:
print("Number of images:", len(features_df))

print("Number of features:", len(features_df.columns))

features_df.head()

Number of images: 249
Number of features: 21


,ID,bug_ratio,shape_symmetry,color_symmetry,R_min,R_max,R_mean,R_median,R_std,G_min,...,G_mean,G_median,G_std,B_min,B_max,B_mean,B_median,B_std,aspect_ratio,circularity
0,1,0.007428,0.994837,32.729621,5,208,68.085745,56.0,47.955399,3,...,54.882747,37.0,45.311197,0,193,39.891969,24.0,36.250747,1.223881,0.037764
1,2,0.008553,0.992870,34.742178,2,248,63.786498,55.0,42.153508,2,...,52.079650,34.0,42.866124,0,244,35.735147,19.0,34.617117,1.514286,0.039630
2,3,0.022093,0.964987,45.799129,3,255,107.176333,114.0,58.771359,0,...,87.794710,86.0,60.057604,0,255,63.383043,52.0,54.823596,1.060606,0.057050
3,4,0.013187,0.973769,46.718097,5,219,87.588243,88.0,46.322587,3,...,71.031852,61.0,46.085760,0,201,50.432535,36.0,37.445621,1.308511,0.035105
4,5,0.009165,0.988398,92.340692,6,255,123.177003,134.0,62.355944,0,...,100.897964,91.0,63.029355,0,245,80.946170,63.0,60.624962,1.200000,0.052363


In [9]:
# ============================================================
# Merge with Labels
# ============================================================

classif_df = pd.read_excel(CLASSIF_PATH)

df = features_df.merge(
    classif_df,
    on="ID",
    how="left"
)

df.head()

,ID,bug_ratio,shape_symmetry,color_symmetry,R_min,R_max,R_mean,R_median,R_std,G_min,...,G_std,B_min,B_max,B_mean,B_median,B_std,aspect_ratio,circularity,bug type,species
0,1,0.007428,0.994837,32.729621,5,208,68.085745,56.0,47.955399,3,...,45.311197,0,193,39.891969,24.0,36.250747,1.223881,0.037764,Bee,Apis mellifera
1,2,0.008553,0.992870,34.742178,2,248,63.786498,55.0,42.153508,2,...,42.866124,0,244,35.735147,19.0,34.617117,1.514286,0.039630,Bee,Apis mellifera
2,3,0.022093,0.964987,45.799129,3,255,107.176333,114.0,58.771359,0,...,60.057604,0,255,63.383043,52.0,54.823596,1.060606,0.057050,Bee,Apis mellifera
3,4,0.013187,0.973769,46.718097,5,219,87.588243,88.0,46.322587,3,...,46.085760,0,201,50.432535,36.0,37.445621,1.308511,0.035105,Bee,Apis mellifera
4,5,0.009165,0.988398,92.340692,6,255,123.177003,134.0,62.355944,0,...,63.029355,0,245,80.946170,63.0,60.624962,1.200000,0.052363,Bee,Apis mellifera


In [10]:
# ============================================================
# Save Features
# ============================================================

df.to_csv(
    "train_features.csv",
    index=False
)

print("train_features.csv saved successfully.")

train_features.csv saved successfully.
